# 06. Feature Set: category

**관점**: '무엇을 선호하는가' - 범주형 통계 요약(엔트로피/집중도/전환빈도)

이 노트북은 **하나의 feature set** 에 대해 5-fold 외부 CV로 AutoGluon(블랙박스 모델)을 학습하고, CV 지표(AUC/LogLoss/Brier)를 확인한 뒤 **OOF 예측**과 **test 예측**을 저장합니다.

- `oof_{setname}.csv` : train OOF (메타모델 학습 입력)
- `oof_test_{setname}.csv` : test 예측 평균 (메타모델 추론 입력)

> 모든 set 노트북은 동일한 `folds.csv` split 을 공유합니다 (스태킹 정합성).

## 0. 설치 & 라이브러리

In [1]:
import sys
!{sys.executable} -m pip install ipython-autotime
!{sys.executable} -m pip install autogluon.tabular gensim

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   ------ --------------------------------- 2.4/15.1 MB 11.2 MB/s eta 0:00:02
   ------------- -------------------------- 5.0/15.1 MB 11.6 MB/s eta 0:00:01
   ------------------ --------------------- 7.1/15.1 MB 11.5 MB/s eta 0:00:01
   ----------------------- ---------------- 8.9/15.1 MB 10.7 MB/s eta 0:00:01
   ---------------------------- ----------- 10.7/15.1 MB 10.5 MB/s eta 0:00:01
   ----------------------------------- ---- 13.4/15.1 MB 10.6 MB/s eta 0:00:01
   ---------------------------------------- 15.1/15.1 MB 10.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.12.3 requires botocore<1.34.70,>=1.34.41, but you have botocore 1.43.23 which is incompatible.


In [3]:
%load_ext autotime

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from autogluon.tabular import TabularPredictor
import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('autogluon').setLevel(logging.ERROR)

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

# ---------------- 전역 설정 ----------------
SEED        = 42
N_SPLITS    = 5
TIME_LIMIT  = 300          # set당 fold 1개 학습 시간(초). 빠른 테스트 시 줄이세요.
PRESETS     = 'good_quality'
EVAL_METRIC = 'roc_auc'    # AutoGluon 내부 모델 선택 기준

DATA_DIR = '../../data/raw'
OOF_DIR  = '../../outputs/oof'
os.makedirs(OOF_DIR, exist_ok=True)

time: 2.48 s (started: 2026-06-05 19:04:38 +09:00)


## 1. 데이터 로드

In [5]:
# 거래 단위 원본 데이터 로드
X_train_trans = pd.read_csv(f'{DATA_DIR}/train_transactions.csv', encoding='cp949')
X_test_trans  = pd.read_csv(f'{DATA_DIR}/test_transactions.csv',  encoding='cp949')
y_train       = pd.read_csv(f'{DATA_DIR}/y_train.csv',            encoding='cp949')

print('train trans:', X_train_trans.shape, '| test trans:', X_test_trans.shape)
print('y_train:', y_train.shape)

train trans: (1036653, 16) | test trans: (689777, 16)
y_train: (30000, 2)
time: 3.16 s (started: 2026-06-05 19:04:40 +09:00)


## 2. Fold split 로드/생성 (공유)

In [7]:
# ===== Fold split: 모든 feature set / 노트북이 공유 =====
# 첫 실행에서 folds.csv 가 없으면 생성, 있으면 로드 -> 7개 set 모두 동일 split 사용 보장
FOLDS_PATH = f'{OOF_DIR}/folds.csv'

if os.path.exists(FOLDS_PATH):
    folds_df = pd.read_csv(FOLDS_PATH)
    print('기존 folds.csv 로드:', folds_df.shape)
else:
    base = y_train[['custid', 'gender']].sort_values('custid').reset_index(drop=True)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    base['fold'] = -1
    for f, (_, val_idx) in enumerate(skf.split(base['custid'], base['gender'])):
        base.loc[val_idx, 'fold'] = f
    folds_df = base[['custid', 'fold']]
    folds_df.to_csv(FOLDS_PATH, index=False)
    print('folds.csv 신규 생성:', folds_df.shape)

print(folds_df['fold'].value_counts().sort_index())

기존 folds.csv 로드: (30000, 2)
fold
0    6000
1    6000
2    6000
3    6000
4    6000
Name: count, dtype: int64
time: 16 ms (started: 2026-06-05 19:04:44 +09:00)


## 3. Feature 생성  ← 팀원이 통째로 교체하는 영역
`features_train`, `features_test` 두 변수를 `custid` + feature 컬럼 형태로 만들면 됩니다.

In [10]:
SETNAME = 'category'

time: 0 ns (started: 2026-06-05 19:04:44 +09:00)


In [11]:
# ===== FEATURE 생성 (수정/교체 가능): Category =====
# 관점: '무엇을 선호하는가' - 범주형 컬럼들의 통계적 요약 (엔트로피/집중도/Top비율/전환빈도)
CAT_COLS = ['brd_nm', 'corner_nm', 'pc_nm', 'part_nm', 'team_nm', 'buyer_nm']

def entropy(s):
    p = s.value_counts(normalize=True)
    return -(p * np.log(p + 1e-12)).sum()
def top_ratio(s):
    return s.value_counts(normalize=True).iloc[0]

def make_features(df):
    df = df.copy()
    df['sales_datetime'] = pd.to_datetime(df['sales_datetime'])
    df = df.sort_values(['custid', 'sales_datetime'])

    out = []
    g = df.groupby('custid')
    for col in CAT_COLS:
        agg = g[col].agg(
            **{f'{col}_nunique': 'nunique',
               f'{col}_entropy': entropy,
               f'{col}_top_ratio': top_ratio}
        )
        out.append(agg)

    # 카테고리 전환 빈도: 연속 거래에서 team_nm 이 바뀐 비율
    def switch_rate(s):
        if len(s) <= 1:
            return 0.0
        return (s.values[1:] != s.values[:-1]).mean()
    sw = g.agg(
        team_switch=('team_nm', switch_rate),
        corner_switch=('corner_nm', switch_rate),
    )
    out.append(sw)

    # 다양성(고유수/거래수)
    cnt = g.size().rename('txn_cnt')
    feats = pd.concat(out + [cnt], axis=1)
    for col in CAT_COLS:
        feats[f'{col}_diversity'] = feats[f'{col}_nunique'] / feats['txn_cnt']
    return feats.fillna(0).reset_index()

features_train = make_features(X_train_trans)
features_test  = make_features(X_test_trans)
print('Category features:', features_train.shape[1] - 1)

# ===== train/test feature 컬럼 정렬 (crosstab 류 카테고리 불일치 방지) =====
feat_cols_ref = [c for c in features_train.columns if c != 'custid']
# test 를 train 컬럼에 맞춤: train 에만 있는 건 0 채움, test 에만 있는 건 버림
features_test = features_test.reindex(columns=['custid'] + feat_cols_ref, fill_value=0)
print('정렬 후 feature 수:', len(feat_cols_ref))

Category features: 27
정렬 후 feature 수: 27
time: 2min 58s (started: 2026-06-05 19:04:44 +09:00)


## 4. 5-fold 외부 CV (AutoGluon = 하나의 블랙박스 모델)

In [16]:
# ===== 5-fold 외부 CV: AutoGluon 을 하나의 블랙박스 모델로 취급 =====
# - 각 fold: train fold 로 AutoGluon fit -> valid fold 예측 = OOF (leakage 없음)
# - test: 5개 fold 모델 예측의 평균 = test meta feature
SETNAME = SETNAME  # 위에서 정의됨

# custid / target / fold 정렬 맞추기
data = features_train.merge(folds_df, on='custid', how='inner') \
                     .merge(y_train[['custid', 'gender']], on='custid', how='inner')
data = data.sort_values('custid').reset_index(drop=True)

feat_cols = [c for c in features_train.columns if c != 'custid']

oof_pred  = np.zeros(len(data))
test_pred = np.zeros(len(features_test))
test_sorted = features_test.sort_values('custid').reset_index(drop=True)

for f in range(N_SPLITS):

    tr_idx = data.index[data['fold'] != f]
    va_idx = data.index[data['fold'] == f]

    train_part = data.loc[tr_idx, feat_cols + ['gender']]
    valid_part = data.loc[va_idx, feat_cols]

    predictor = TabularPredictor(
        label='gender', problem_type='binary',
        eval_metric=EVAL_METRIC, verbosity=0,
    ).fit(train_data=train_part, presets=PRESETS, time_limit=TIME_LIMIT)

    pos = predictor.positive_class
    oof_pred[va_idx] = predictor.predict_proba(valid_part)[pos].values
    test_pred += predictor.predict_proba(test_sorted[feat_cols])[pos].values / N_SPLITS

    fold_auc = roc_auc_score(data.loc[va_idx, 'gender'], oof_pred[va_idx])
    print(f'[fold {f}] valid AUC = {fold_auc:.5f}')

		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick ti

[fold 0] valid AUC = 0.58522


		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick ti

[fold 1] valid AUC = 0.58501


		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick ti

[fold 2] valid AUC = 0.57207


		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick ti

[fold 3] valid AUC = 0.57830


		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick ti

[fold 4] valid AUC = 0.59369
time: 22min 56s (started: 2026-06-05 19:07:42 +09:00)


## 5. CV 지표

In [18]:
# ===== CV 지표: AUC(순위) + LogLoss/Brier(칼리브레이션·오답 페널티) =====
y_true = data['gender'].values
cv_auc   = roc_auc_score(y_true, oof_pred)
cv_ll    = log_loss(y_true, oof_pred)
cv_brier = brier_score_loss(y_true, oof_pred)

print(f'=== {SETNAME} CV ===')
print(f'AUC      : {cv_auc:.5f}   (순위 품질, 높을수록 좋음)')
print(f'LogLoss  : {cv_ll:.5f}   (확신한 오답에 큰 페널티, 낮을수록 좋음)')
print(f'Brier    : {cv_brier:.5f}   (확률 칼리브레이션, 낮을수록 좋음)')

=== category CV ===
AUC      : 0.58257   (순위 품질, 높을수록 좋음)
LogLoss  : 0.60458   (확신한 오답에 큰 페널티, 낮을수록 좋음)
Brier    : 0.20743   (확률 칼리브레이션, 낮을수록 좋음)
time: 78 ms (started: 2026-06-05 19:30:39 +09:00)


## 6. OOF / test 예측 저장

In [22]:
# ===== OOF / test 예측 저장 (스키마: custid, pred_proba) =====
oof_out = pd.DataFrame({
    'custid': data['custid'].values,
    'pred_proba': oof_pred,
    'fold': data['fold'].values,
})
oof_out.to_csv(f'{OOF_DIR}/oof_{SETNAME}_0.csv', index=False)

test_out = pd.DataFrame({
    'custid': test_sorted['custid'].values,
    'pred_proba': test_pred,
})
test_out.to_csv(f'{OOF_DIR}/test_{SETNAME}_0.csv', index=False)

print('저장 완료:')
print(f'  {OOF_DIR}/oof_{SETNAME}_0.csv      ', oof_out.shape)
print(f'  {OOF_DIR}/oof_test_{SETNAME}_0.csv ', test_out.shape)
display(oof_out.head())

저장 완료:
  ../../outputs/oof/oof_category.csv       (30000, 3)
  ../../outputs/oof/oof_test_category.csv  (19995, 2)


,custid,pred_proba,fold
0,0,0.232487,3
1,1,0.321339,1
2,2,0.337024,0
3,3,0.332476,0
4,4,0.266029,0


time: 63 ms (started: 2026-06-05 19:30:58 +09:00)
